The main objective of this project is to develop a robust and accurate predective model for use in diagnosis based on specific markers or features.

In [103]:
import pandas as pd
import numpy as np

## Chronic kidney disease

In [104]:
# first let's import the dataset and explore it

df = pd.read_csv('chronic_kindey_disease.csv', na_values='?') # missing values are denoted by ? - let's replace them with pandas NaN values
df.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,status
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,no,no,no,good,no,no,ckd


In [105]:
# to understand better the columns, let's look at the info file of the dataset

with open("chronic_kidney_disease_info.txt", "r") as f:
    content = f.read()

print(content)

% 1. Title: Early stage of Indians Chronic Kidney Disease(CKD)
%
% 2. Source Information:
%   (a) Source: 
			Dr.P.Soundarapandian.M.D.,D.M
			(Senior Consultant Nephrologist), 
			Apollo  Hospitals, 
			Managiri,
			Madurai Main Road, 
			Karaikudi,
			Tamilnadu,
			India.

%   (b) Creator: 
			L.Jerlin Rubini(Research Scholar)
			Alagappa University,
			EmailId   :jel.jerlin@gmail.com
			ContactNo :+91-9597231281

%   (c) Guided by: 
			Dr.P.Eswaran Assistant Professor,
			Department of Computer Science and Engineering,
			Alagappa University,
			Karaikudi,
			Tamilnadu,
			India.
			Emailid:eswaranperumal@gmail.com

%   (d) Date     : July 2015
%
% 3.Relevant Information:
			age		-	age	
			bp		-	blood pressure
			sg		-	specific gravity
			al		-   	albumin
			su		-	sugar
			rbc		-	red blood cells
			pc		-	pus cell
			pcc		-	pus cell clumps
			ba		-	bacteria
			bgr		-	blood glucose random
			bu		-	blood urea
			sc		-	serum creatinine
			sod		-	sodium
			pot		-	potassium
			hemo		-	hem

We see there are 25 columns and 400 rows. Out of the 25 columns, 11 are numeric and 14 are nominal. Out of the 400 rows, there are 250 cases of chronic kindey disease and 150 cases of not CKD. The "patient identifier" is their age (even though this is not a unique value), and then we have the results of several lab tests of this patient, and then in the 'status' column whether the patient has CKD or not.

In [106]:
# we will also replace yes by 1 and no by 0, and present by 1 and notpresent by 0, etc.

df['rbc'] = df['rbc'].replace({'normal':1, 'abnormal':0})
df['pc'] = df['pc'].replace({'normal':1, 'abnormal':0})
df['pcc'] = df['pcc'].replace({'notpresent':0, 'present':1})
df['ba'] = df['ba'].replace({'notpresent':0, 'present':1})

df['htn'] = df['htn'].replace({'no':0, 'yes':1})
df['dm'] = df['dm'].replace({'\tno':'no'})
df['dm'] = df['dm'].replace({'no':0, 'yes':1})
df['cad'] = df['cad'].replace({'no':0, 'yes':1})
df['appet'] = df['appet'].replace({'poor':0, 'good':1})
df['pe'] = df['pe'].replace({'no':0, 'yes':1})
df['ane'] = df['ane'].replace({'no':0, 'yes':1})

df['status'] = df['status'].replace({'ckd\t':'ckd'})
df['status'] = df['status'].replace({'notckd':0, 'ckd':1})

df.head()

C:\Users\ckrigul\AppData\Local\Temp\ipykernel_6156\1064134437.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['rbc'] = df['rbc'].replace({'normal':1, 'abnormal':0})
C:\Users\ckrigul\AppData\Local\Temp\ipykernel_6156\1064134437.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['pc'] = df['pc'].replace({'normal':1, 'abnormal':0})
C:\Users\ckrigul\AppData\Local\Temp\ipykernel_6156\1064134437.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old beha

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,status
0,48.0,80.0,1.020,1.0,0.0,NaN,1.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,1.0,0.0,0.0,1
1,7.0,50.0,1.020,4.0,0.0,NaN,1.0,0.0,0.0,NaN,...,38.0,6000.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,1
2,62.0,80.0,1.010,2.0,3.0,1.0,1.0,0.0,0.0,423.0,...,31.0,7500.0,NaN,0.0,1.0,0.0,0.0,0.0,1.0,1
3,48.0,70.0,1.005,4.0,0.0,1.0,0.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,0.0,1.0,1.0,1
4,51.0,80.0,1.010,2.0,0.0,1.0,1.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,1.0,0.0,0.0,1


In [107]:
# let's look at all numeric columns individually and see if we find something peculiar

df = df.astype('float') # changing column types to integer
df.describe()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,status
count,391.000000,388.000000,353.000000,354.000000,351.000000,248.000000,335.000000,396.000000,396.000000,356.000000,...,329.000000,294.000000,269.000000,398.000000,398.000000,398.000000,399.000000,399.000000,399.000000,400.000000
mean,51.483376,76.469072,1.017408,1.016949,0.450142,0.810484,0.773134,0.106061,0.055556,148.036517,...,38.884498,8406.122449,4.707435,0.369347,0.344221,0.085427,0.794486,0.190476,0.150376,0.625000
std,17.169714,13.683637,0.005717,1.352679,1.099191,0.392711,0.419431,0.308305,0.229351,79.281714,...,8.990105,2944.474190,1.025323,0.483235,0.475712,0.279868,0.404584,0.393170,0.357888,0.484729
min,2.000000,50.000000,1.005000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22.000000,...,9.000000,2200.000000,2.100000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,42.000000,70.000000,1.010000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,99.000000,...,32.000000,6500.000000,3.900000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
50%,55.000000,80.000000,1.020000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,121.000000,...,40.000000,8000.000000,4.800000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,1.000000
75%,64.500000,80.000000,1.020000,2.000000,0.000000,1.000000,1.000000,0.000000,0.000000,163.000000,...,45.000000,9800.000000,5.400000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000
max,90.000000,180.000000,1.025000,5.000000,5.000000,1.000000,1.000000,1.000000,1.000000,490.000000,...,54.000000,26400.000000,8.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [108]:
# now we look at the number of missing values per column
# we see some columns with even more than 25% missing data

df.isna().sum()

age         9
bp         12
sg         47
al         46
su         49
rbc       152
pc         65
pcc         4
ba          4
bgr        44
bu         19
sc         17
sod        87
pot        88
hemo       52
pcv        71
wbcc      106
rbcc      131
htn         2
dm          2
cad         2
appet       1
pe          1
ane         1
status      0
dtype: int64

These missing values could be handled in many ways - we could remove the rows with missing values, or we could impute the values. However at this stage, we will leave them as they are, as the later approach depends on the specific features and ML model chosen.

In [100]:
# next, we are interested in if there are any outliers
# one way to do this is using the standard interquantile range on the numerical columns

def remove_outliers(df, col):
    
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
        
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df_clean = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
        
    return df_clean

In [101]:
# (we see that all 10 detected outliers of the age column are children under the age of 9
# we will proceed to remove these observations)

df_clean = remove_outliers(df, 'age')
df_clean = remove_outliers(df_clean, 'bp')
df_clean = remove_outliers(df_clean, 'bgr')
df_clean = remove_outliers(df_clean, 'bu')
df_clean = remove_outliers(df_clean, 'sc')
df_clean = remove_outliers(df_clean, 'sod')
df_clean = remove_outliers(df_clean, 'pot')
df_clean = remove_outliers(df_clean, 'hemo')
df_clean = remove_outliers(df_clean, 'pcv')
df_clean = remove_outliers(df_clean, 'wbcc')
df_clean = remove_outliers(df_clean, 'rbcc')

# left with 145 rows out of the original 400

It will be interesting to see how df and df_clean compare in the next analysis and the prediction of CKD.